[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C43_Data_Engineering_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy / pandas / 标准库、CPU 可跑**，用 **小规模真实模拟** 复现 PB 级数据工程的机制，再用 **复杂度账 + 吞吐账** 把规模问题推演出来。

这个 notebook 做三件事：① 确认环境；② 用一个最小例子体会「**规模的断崖**」——为什么两两比对去重在大规模直接不可行；③ 立下全课的两条纪律——**对拍（differential testing）** 与 **算账（accounting）**。

## 1 · 环境自检

只需要 `numpy` 与 `pandas`；`hashlib`/`re`/`collections` 是标准库。`matplotlib` 可选。

In [ ]:
import sys, platform, hashlib, re, collections
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np;  print('numpy', np.__version__)
import pandas as pd; print('pandas', pd.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 复杂度账：两两比对去重的「断崖」

去重最朴素的做法是把每篇文档和其他所有文档比一遍，复杂度 **O(n²)**（确切是 C(n,2) = n(n−1)/2 次比较）。

在小数据上这毫无问题。下面把它跑通、确认正确，**再算一笔账**：同一个算法在 1 万 / 1 亿 / 1 万亿篇文档上分别要多久？

In [ ]:
def brute_force_pair_count(n):
    '''两两比对的比较次数 = C(n,2)。'''
    return n * (n - 1) // 2

# 先在小数据上确认它确实是 O(n^2)：实际跑一遍计数
def brute_force_dedup_demo(docs):
    '''最朴素去重：两两比相等，返回保留的下标与实际比较次数。'''
    keep, seen, comparisons = [], [], 0
    for i, d in enumerate(docs):
        dup = False
        for j in keep:
            comparisons += 1
            if docs[j] == d:
                dup = True; break
        if not dup:
            keep.append(i)
    return keep, comparisons

docs = ['a', 'b', 'a', 'c', 'b', 'a']
keep, comps = brute_force_dedup_demo(docs)
assert [docs[i] for i in keep] == ['a', 'b', 'c'], '应保留去重后的唯一文档'
print('小数据去重正确，保留:', [docs[i] for i in keep])

PER_SEC = 1e9   # 乐观假设：每秒 10 亿次比较
print(f"\n{'文档数 n':>14s} {'比较次数 C(n,2)':>20s} {'单核耗时':>16s}")
for n in [1e4, 1e6, 1e8, 1e12]:
    pairs = brute_force_pair_count(int(n))
    secs = pairs / PER_SEC
    years = secs / (3600 * 24 * 365)
    human = f'{secs:.2g} 秒' if secs < 3600 else f'{years:.2g} 年'
    print(f'{n:>14.0e} {pairs:>20.2e} {human:>16s}')
print('\n断崖：n 每涨 100 倍，比较次数涨 1e4 倍。1e12 篇 -> 约 1.6e7 年。')
print('结论：O(n^2) 在大规模不是「慢」，是「不可行」。模块 01 用 LSH 把它降到近 O(n)。')

**关键结论**：O(n²) 算法在大规模会**断崖式**地从可行掉进不可行——优化常数没用，必须换成近 O(n) 的算法（MinHash+LSH，模块 01）。

这就是本课反复强调的第一性原理：**先用复杂度账判生死，再谈优化常数**。

## 3 · 吞吐账：万亿 token 分词要多久？

分词（tokenization）逐文档独立、是**易并行**的。但万亿（1e12）token 的量级下，吞吐照样是硬约束。

做一笔吞吐账：单核每秒能分 `R` 个 token，用 `P` 个核、并行效率 `eff`，分完 1e12 token 要多久？

In [ ]:
def tokenize_time(total_tokens, per_core_tok_per_s, cores, eff=0.8):
    '''返回分完所需秒数。eff 是并行效率(<1，因 IO/调度损耗)。'''
    effective_rate = per_core_tok_per_s * cores * eff
    return total_tokens / effective_rate

TOTAL = 1e12                      # 1 万亿 token
R = 1e5                           # 单核每秒 ~10 万 token（典型量级）
print(f"{'核数':>8s} {'有效吞吐(tok/s)':>18s} {'耗时':>14s}")
for cores in [1, 64, 1024]:
    secs = tokenize_time(TOTAL, R, cores)
    days = secs / (3600 * 24)
    human = f'{secs/3600:.1f} 小时' if days < 1 else f'{days:.1f} 天'
    print(f'{cores:>8d} {R*cores*0.8:>18.2e} {human:>14s}')
# 单核要几个月；上千核才能压到天级。这就是为什么分词必须并行（模块 03）。
secs_1 = tokenize_time(TOTAL, R, 1)
secs_1024 = tokenize_time(TOTAL, R, 1024)
assert secs_1 / secs_1024 > 800, '1024 核应带来近 1024x（打了效率折扣）的加速'
print('\n✅ 吞吐账把「快/慢」变成可规划的工程数字：要天级完成，得上千核。')

## 4 · 立纪律一：对拍（differential testing）

本课每个**工程化的近似/加速版**都要和一个**暴力/朴素参考**比对。比如一个「用哈希集合去重」的 O(n) 实现，必须和上面 O(n²) 的暴力去重给出**相同结果**。

先把这个工作流跑通。

In [ ]:
def hash_dedup(docs):
    '''O(n) 精确去重：用集合记录已见内容哈希。返回保留下标。'''
    seen, keep = set(), []
    for i, d in enumerate(docs):
        h = hashlib.sha1(d.encode('utf-8')).hexdigest()
        if h not in seen:
            seen.add(h); keep.append(i)
    return keep

rng = np.random.default_rng(0)
vocab = ['alpha', 'beta', 'gamma', 'delta']
docs = [vocab[k] for k in rng.integers(0, len(vocab), size=200)]

keep_brute = brute_force_dedup_demo(docs)[0]
keep_hash = hash_dedup(docs)
# 对拍：两种方法保留的「内容集合」必须一致（顺序可能不同，比集合）
assert set(docs[i] for i in keep_brute) == set(docs[i] for i in keep_hash)
assert len(keep_hash) == len(set(docs)), '精确去重后应等于唯一文档数'
print(f'暴力 O(n^2) 保留 {len(keep_brute)} 篇；哈希 O(n) 保留 {len(keep_hash)} 篇 -> 一致 ✅')
print('对拍通过：O(n) 的工程版与 O(n^2) 的朴素参考结果相同 —— 这就是全课的正确性纪律。')

## 5 · 立纪律二：算账（accounting）封装

把「复杂度账」和「吞吐账」各封装成一个小工具，后面每个模块都会用它把机制推演到 PB 级。

In [ ]:
def feasibility(n, per_op_complexity, ops_per_sec=1e9):
    '''给定规模 n 与复杂度函数，估算耗时(秒)。
       per_op_complexity(n) 返回操作总数。'''
    total_ops = per_op_complexity(n)
    return total_ops / ops_per_sec

quadratic = lambda n: n * (n - 1) / 2     # O(n^2)
linear    = lambda n: n                    # O(n)

n = int(1e12)
t_quad = feasibility(n, quadratic)
t_lin  = feasibility(n, linear)
print(f'1e12 文档：O(n^2) 需 {t_quad/3.15e7:.2e} 年；O(n) 需 {t_lin:.2e} 秒 = {t_lin/3600:.1f} 小时')
speedup = t_quad / t_lin
assert speedup > 1e10, 'O(n) 相对 O(n^2) 在 1e12 规模应快 10 个数量级以上'
print(f'O(n) 比 O(n^2) 快约 {speedup:.1e} 倍 —— 这不是优化，是换了一类算法 ✅')
print('\n本课契约：每个机制都配 复杂度账 + 吞吐账 + 对拍参考，把「规模可行性」量化到可检验。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在小规模真实数据上写出的每个去重/加载/分词/过滤/去污染机制，都会 ① 与暴力/朴素参考**对拍**确认正确，② 用复杂度/吞吐**算账**推演到 PB 级。结构正确 + 账目过关 = 这套工程能真正扩展。

**接下来六个模块**：01 大规模去重 → 02 流式加载与分片 → 03 Tokenization 吞吐 → 04 大规模质量过滤 → 05 溯源与去污染。

下一站：**模块 01 · 大规模去重** —— 把 O(n²) 的墙摆在你面前，再用 MinHash+LSH 把它推倒。